In [6]:
from jupyter_dash import JupyterDash
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import base64
import pandas as pd

from AnimalShelter import AnimalShelter

JupyterDash.infer_jupyter_proxy_config()

username = "aacuser"
password = "Password123"

db = AnimalShelter(username, password)

df = pd.DataFrame.from_records(db.read({}))
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

app = JupyterDash(__name__)

image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    html.Center([
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'height': '120px'}
        ),
        html.H1('Grazioso Salvare Rescue Animal Dashboard'),
        html.H3('Elizabeth Wing - CS 340 Project Two')
    ]),

    html.Hr(),

    html.Div([
        html.Label("Select Rescue Type:"),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'wilderness'},
                {'label': 'Disaster or Individual Tracking', 'value': 'tracking'},
                {'label': 'Reset', 'value': 'reset'}
            ],
            value='reset',
            labelStyle={'display': 'inline-block', 'marginRight': '20px'}
        )
    ]),

    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True}
            for i in df.columns
        ],
        data=df.to_dict('records'),
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[0],
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'minWidth': '100px',
            'width': '150px',
            'maxWidth': '200px',
            'whiteSpace': 'normal'
        }
    ),

    html.Br(),
    html.Hr(),

    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(id='graph-id', className='col s12 m6'),
            html.Div(id='map-id', className='col s12 m6')
        ]
    )
])


@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):

    if filter_type == 'water':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "Labrador Retriever Mix",
                "Chesapeake Bay Retriever",
                "Newfoundland"
            ]},
            "sex_upon_outcome": "Intact Female",
            "$and": [
                {"age_upon_outcome_in_weeks": {"$gte": 26}},
                {"age_upon_outcome_in_weeks": {"$lte": 156}}
            ]
        }

    elif filter_type == 'wilderness':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "German Shepherd",
                "Alaskan Malamute",
                "Old English Sheepdog",
                "Siberian Husky",
                "Rottweiler"
            ]},
            "sex_upon_outcome": "Intact Male",
            "$and": [
                {"age_upon_outcome_in_weeks": {"$gte": 26}},
                {"age_upon_outcome_in_weeks": {"$lte": 156}}
            ]
        }

    elif filter_type == 'tracking':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "Doberman Pinscher",
                "German Shepherd",
                "Golden Retriever",
                "Golden Retriever Mix",
                "Bloodhound",
                "Rottweiler"
            ]},
            "sex_upon_outcome": "Intact Male",
            "$and": [
                {"age_upon_outcome_in_weeks": {"$gte": 20}},
                {"age_upon_outcome_in_weeks": {"$lte": 300}}
            ]
        }

    else:
        query = {}

    dff = pd.DataFrame.from_records(db.read(query))

    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    return dff.to_dict('records')


@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return [html.H3("No data available")]

    dff = pd.DataFrame.from_dict(viewData)

    return [
        dcc.Graph(
            figure=px.pie(
                dff,
                names='breed',
                title='Animal Breeds Displayed in Current Results'
            )
        )
    ]


@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


@app.callback(
    Output('map-id', "children"),
    [
        Input('datatable-id', "derived_virtual_data"),
        Input('datatable-id', "derived_virtual_selected_rows")
    ]
)
def update_map(viewData, index):
    if viewData is None or len(viewData) == 0:
        return [html.H3("No map data available")]

    dff = pd.DataFrame.from_dict(viewData)

    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[
                        dff.iloc[row]['location_lat'],
                        dff.iloc[row]['location_long']
                    ],
                    children=[
                        dl.Tooltip(dff.iloc[row]['breed']),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(dff.iloc[row]['name'])
                        ])
                    ]
                )
            ]
        )
    ]


app.run_server(mode='inline', port=8050)

---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
TypeError: 'NoneType' object is not iterable

